# Notebook 03 — Structural Harmonization

Loads MedQuAD and MedDialog-EN, inspects schemas, and converts both into a common turn-level structure.

**Actual file paths:**
- MedQuAD: `data/raw/medquad.csv` — 16,412 rows, columns: `question`, `answer`, `source`, `focus_area`
- MedDialog: `data/raw/meddialog/HealthCareMagic-100k.json` — 112,165 rows, columns: `instruction`, `input`, `output`

HealthChat-11K excluded: local file contains metadata only (no utterance text).

In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
tqdm.pandas()

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed/structural')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('Output directory:', PROCESSED_DIR.resolve())

Output directory: C:\Users\nirmi\Desktop\Capstone\data\processed\structural


## 1. MedQuAD — Schema Inspection

In [2]:
medquad_raw = pd.read_csv(RAW_DIR / 'medquad.csv')

print('=== MedQuAD ===')
print('Shape:', medquad_raw.shape)
print('Columns:', medquad_raw.columns.tolist())
print()
print('Dtypes:')
print(medquad_raw.dtypes)
print()
print('Missing values:')
print(medquad_raw.isnull().sum())
print()
print('First 5 records:')
display(medquad_raw.head())

=== MedQuAD ===
Shape: (16412, 4)
Columns: ['question', 'answer', 'source', 'focus_area']

Dtypes:
question      str
answer        str
source        str
focus_area    str
dtype: object

Missing values:
question       0
answer         5
source         0
focus_area    14
dtype: int64

First 5 records:


,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma


In [3]:
print('Available labels (focus_area):')
print(f'  Unique values: {medquad_raw["focus_area"].nunique()}')
print(medquad_raw['focus_area'].value_counts().head(20))
print()
print('Available labels (source):')
print(medquad_raw['source'].value_counts())
print()
print('Example record:')
ex = medquad_raw.iloc[0]
print('  Q:', ex['question'])
print('  A:', str(ex['answer'])[:300])
print('  source:', ex['source'])
print('  focus_area:', ex['focus_area'])

Available labels (focus_area):
  Unique values: 5126
focus_area
Breast Cancer                       53
Prostate Cancer                     43
Stroke                              35
Skin Cancer                         34
Alzheimer's Disease                 30
Colorectal Cancer                   29
Lung Cancer                         29
High Blood Cholesterol              28
Heart Attack                        28
Heart Failure                       28
Causes of Diabetes                  28
High Blood Pressure                 27
Parkinson's Disease                 25
Leukemia                            22
Osteoporosis                        21
Shingles                            21
Diabetes                            20
Age-related Macular Degeneration    20
Hemochromatosis                     20
Diabetic Retinopathy                19
Name: count, dtype: int64

Available labels (source):
source
GHR                  5430
GARD                 5394
NIDDK                1192
NINDS            

## 2. MedDialog-EN — Schema Inspection

In [4]:
# Stored as raw JSON: data/raw/meddialog/HealthCareMagic-100k.json
meddialog_path = RAW_DIR / 'meddialog' / 'HealthCareMagic-100k.json'
meddialog_df = pd.read_json(meddialog_path)

print('=== MedDialog-EN ===')
print('File:', meddialog_path.name)
print('Shape:', meddialog_df.shape)
print('Columns:', meddialog_df.columns.tolist())
print()
print('Dtypes:')
print(meddialog_df.dtypes)
print()
print('Missing values:')
print(meddialog_df.isnull().sum())
print()
print('First 5 records:')
display(meddialog_df.head())

=== MedDialog-EN ===
File: HealthCareMagic-100k.json
Shape: (112165, 3)
Columns: ['instruction', 'input', 'output']

Dtypes:
instruction    str
input          str
output         str
dtype: object

Missing values:
instruction    0
input          0
output         0
dtype: int64

First 5 records:


,instruction,input,output
0,"If you are a doctor, please answer the medical...",I woke up this morning feeling the whole room ...,"Hi, Thank you for posting your query. The most..."
1,"If you are a doctor, please answer the medical...",My baby has been pooing 5-6 times a day for a ...,Hi... Thank you for consulting in Chat Doctor....
2,"If you are a doctor, please answer the medical...","Hello, My husband is taking Oxycodone due to a...","Hello, and I hope I can help you today.First, ..."
3,"If you are a doctor, please answer the medical...",lump under left nipple and stomach pain (male)...,HI. You have two different problems. The lump ...
4,"If you are a doctor, please answer the medical...",I have a 5 month old baby who is very congeste...,Thank you for using Chat Doctor. I would sugge...


In [5]:
print('Unique instruction values:', meddialog_df['instruction'].nunique())
print(meddialog_df['instruction'].value_counts())
print()
print('Example dialogue:')
ex = meddialog_df.iloc[0]
print('  instruction:', ex['instruction'])
print('  input:', ex['input'][:300])
print('  output:', ex['output'][:300])

Unique instruction values: 1
instruction
If you are a doctor, please answer the medical questions based on the patient's description.    112165
Name: count, dtype: int64

Example dialogue:
  instruction: If you are a doctor, please answer the medical questions based on the patient's description.
  input: I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or 
  output: Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo (BPPV), a type of peripheral vertigo. In this condition, the most common symptom is dizziness or giddiness, which is made worse with movements. Accompanying nausea and vomiting are c


## 3. Structural Harmonization

Both datasets are QA pairs → converted to 2-turn **constructed** dialogues:
- turn 0: `speaker=user`, `utterance=question/input`
- turn 1: `speaker=assistant`, `utterance=answer/output`

In [6]:
def harmonize_medquad(df):
    records = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc='MedQuAD'):
        did = f'medquad_{idx:06d}'
        label = row['focus_area'] if pd.notna(row.get('focus_area')) else None
        records.append({'dialogue_id': did, 'turn_id': 0, 'speaker': 'user',
                        'utterance': str(row['question']) if pd.notna(row['question']) else None,
                        'source_dataset': 'MedQuAD', 'original_id': str(idx),
                        'source_label_raw': label, 'dialogue_origin': 'constructed'})
        records.append({'dialogue_id': did, 'turn_id': 1, 'speaker': 'assistant',
                        'utterance': str(row['answer']) if pd.notna(row['answer']) else None,
                        'source_dataset': 'MedQuAD', 'original_id': str(idx),
                        'source_label_raw': label, 'dialogue_origin': 'constructed'})
    return pd.DataFrame(records)

def harmonize_meddialog(df):
    records = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc='MedDialog'):
        did = f'meddialog_{idx:07d}'
        label = str(row['instruction'])[:200] if pd.notna(row.get('instruction')) else None
        records.append({'dialogue_id': did, 'turn_id': 0, 'speaker': 'user',
                        'utterance': str(row['input']) if pd.notna(row['input']) else None,
                        'source_dataset': 'MedDialog', 'original_id': str(idx),
                        'source_label_raw': label, 'dialogue_origin': 'constructed'})
        records.append({'dialogue_id': did, 'turn_id': 1, 'speaker': 'assistant',
                        'utterance': str(row['output']) if pd.notna(row['output']) else None,
                        'source_dataset': 'MedDialog', 'original_id': str(idx),
                        'source_label_raw': label, 'dialogue_origin': 'constructed'})
    return pd.DataFrame(records)

medquad_turns  = harmonize_medquad(medquad_raw)
meddialog_turns = harmonize_meddialog(meddialog_df)

print('MedQuAD turns:', medquad_turns.shape)
print('MedDialog turns:', meddialog_turns.shape)

MedQuAD:   0%|          | 0/16412 [00:00<?, ?it/s]

MedDialog:   0%|          | 0/112165 [00:00<?, ?it/s]

MedQuAD turns: (32824, 8)
MedDialog turns: (224330, 8)


In [7]:
STRUCTURAL_COLS = ['dialogue_id', 'turn_id', 'speaker', 'utterance',
                   'source_dataset', 'original_id', 'source_label_raw', 'dialogue_origin']

harmonized = pd.concat([medquad_turns, meddialog_turns], ignore_index=True)
harmonized = harmonized[STRUCTURAL_COLS]

assert harmonized.duplicated(subset=['dialogue_id', 'turn_id']).sum() == 0
assert set(harmonized['speaker'].unique()) <= {'user', 'assistant'}
print('All checks passed.')
print('Combined shape:', harmonized.shape)
display(harmonized.head(6))

All checks passed.
Combined shape: (257154, 8)


,dialogue_id,turn_id,speaker,utterance,source_dataset,original_id,source_label_raw,dialogue_origin
0,medquad_000000,0,user,What is (are) Glaucoma ?,MedQuAD,0,Glaucoma,constructed
1,medquad_000000,1,assistant,Glaucoma is a group of diseases that can damag...,MedQuAD,0,Glaucoma,constructed
2,medquad_000001,0,user,What causes Glaucoma ?,MedQuAD,1,Glaucoma,constructed
3,medquad_000001,1,assistant,"Nearly 2.7 million people have glaucoma, a lea...",MedQuAD,1,Glaucoma,constructed
4,medquad_000002,0,user,What are the symptoms of Glaucoma ?,MedQuAD,2,Glaucoma,constructed
5,medquad_000002,1,assistant,Symptoms of Glaucoma Glaucoma can develop in ...,MedQuAD,2,Glaucoma,constructed


In [8]:
harmonized.to_parquet(PROCESSED_DIR / 'harmonized_structural.parquet', index=False)
harmonized.to_csv(PROCESSED_DIR / 'harmonized_structural.csv', index=False)
print('Saved.')

print('\n=== SUMMARY ===')
for ds in ['MedQuAD', 'MedDialog']:
    s = harmonized[harmonized['source_dataset'] == ds]
    print(f'{ds}: {s["dialogue_id"].nunique():,} dialogues, {len(s):,} turns')
print(f'Total: {harmonized["dialogue_id"].nunique():,} dialogues, {len(harmonized):,} turns')
print(f'User turns: {(harmonized["speaker"]=="user").sum():,}')
print(f'Assistant turns: {(harmonized["speaker"]=="assistant").sum():,}')

Saved.

=== SUMMARY ===
MedQuAD: 16,412 dialogues, 32,824 turns
MedDialog: 112,165 dialogues, 224,330 turns
Total: 128,577 dialogues, 257,154 turns
User turns: 128,577
Assistant turns: 128,577
